# Flux handwriting pilot v3: restyle a clean printed word into real handwriting

Last run showed the model learns the handwriting look but invents the letters. This run gives it the clean printed word as input, so the spelling is fixed by the input and the model only changes the style.

You need `edit_pairs.zip`, made by `tools/prepare_flux_edit_pairs.py`. Put it in the Colab Files panel or at the top level of Google Drive.

Before you start: Runtime, Change runtime type, pick a GPU (L4 if you can, else T4). Run the cells in order and keep the tab open. Checkpoints go to Google Drive (`flux_out_edit`).

If a cell fails, send me the error text.

In [ ]:
# 1) GPU check and Google Drive
import subprocess, shutil
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
else:
    print('NO GPU FOUND. Use Runtime > Change runtime type and pick a GPU, then run this cell again.')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Load the training pairs
import os, glob, subprocess
zp = '/content/edit_pairs.zip' if os.path.exists('/content/edit_pairs.zip') else '/content/drive/MyDrive/edit_pairs.zip'
print('using', zp)
subprocess.run('rm -rf /content/edit_pairs /content/edit_raw', shell=True)
subprocess.run(['unzip', '-q', '-o', zp, '-d', '/content/edit_raw'], check=True)
root = os.path.dirname(glob.glob('/content/edit_raw/**/instruction.txt', recursive=True)[0])
subprocess.run(['mv', root, '/content/edit_pairs'], check=True)
for d in ('reference', 'target', 'test_reference'):
    print(d, len(glob.glob(f'/content/edit_pairs/{d}/*.png')), 'images')
print('captions:', len(glob.glob('/content/edit_pairs/target/*.txt')))
print('instruction:', open('/content/edit_pairs/instruction.txt', encoding='utf8').read())

In [ ]:
# 3) Install the trainer (ai-toolkit). Safe to run again.
%cd /content
import os
if not os.path.isdir('/content/ai-toolkit'):
    !git clone https://github.com/ostris/ai-toolkit /content/ai-toolkit
%cd /content/ai-toolkit
# Colab runs Python 3.13 and ai-toolkit pins an old scipy that has no ready-made build: drop that pin.
!sed -i -E 's/^scipy[^#]*/scipy/' requirements.txt
!pip install -q -r requirements.txt
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4) Training settings: klein 4B restyle LoRA.
# STEPS: 500 is a first try. BFL's guide says edit LoRAs usually peak between 1000 and 1750 steps,
# so raise it if you have a faster GPU (an L4 should do 1000 in a few hours).
STEPS = 500
import yaml
cfg = yaml.safe_load(open('/content/ai-toolkit/config/examples/train_lora_flux_24gb.yaml'))
cfg['config']['name'] = 'hwdv1_edit_pilot'
proc = cfg['config']['process'][0]
proc['training_folder'] = '/content/drive/MyDrive/flux_out_edit'
proc.pop('trigger_word', None)
proc['network']['linear'] = 32
proc['network']['linear_alpha'] = 32
ds = proc['datasets'][0]
ds['folder_path'] = '/content/edit_pairs/target'      # real handwriting (output)
ds['control_path'] = '/content/edit_pairs/reference'  # clean printed word (input)
ds['caption_ext'] = 'txt'
ds['caption_dropout_rate'] = 0.0
ds['resolution'] = [512]
proc['train']['steps'] = STEPS
proc['train']['lr'] = 1e-4
proc['train']['skip_first_sample'] = True
proc['train']['disable_sampling'] = True
proc['save']['save_every'] = 250
proc['model'] = {'arch': 'flux2_klein_4b',
                 'name_or_path': 'black-forest-labs/FLUX.2-klein-base-4B',
                 'quantize': True, 'quantize_te': True, 'low_vram': True}
proc['sample']['sample_every'] = 1000000   # no sample pictures during training; we test with our own cell below
proc['sample']['width'] = 512
proc['sample']['height'] = 512
yaml.safe_dump(cfg, open('/content/ai-toolkit/config/hwdv1_edit_pilot.yaml', 'w'), allow_unicode=True)
print(yaml.safe_dump(cfg, allow_unicode=True)[:3000])

In [ ]:
# 5) Train. The progress line shows e.g. 19/500. Checkpoints are saved to Drive: flux_out_edit/hwdv1_edit_pilot
%cd /content/ai-toolkit
!python run.py config/hwdv1_edit_pilot.yaml

In [ ]:
# 6) Restyle the 12 test words (clean printed word in, handwriting out)
import torch, glob, os
from PIL import Image
try:
    from diffusers import Flux2KleinPipeline
except ImportError:
    !pip install -q -U git+https://github.com/huggingface/diffusers
    from diffusers import Flux2KleinPipeline
out_dir = '/content/drive/MyDrive/flux_out_edit/hwdv1_edit_pilot'
final = os.path.join(out_dir, 'hwdv1_edit_pilot.safetensors')
files = sorted(glob.glob(out_dir + '/*.safetensors'))
lora = final if os.path.exists(final) else files[-1]
print('using', lora)
instruction = open('/content/edit_pairs/instruction.txt', encoding='utf8').read().strip()
words = open('/content/edit_pairs/test_words.txt', encoding='utf8').read().split('\n')
pipe = Flux2KleinPipeline.from_pretrained('black-forest-labs/FLUX.2-klein-4B', torch_dtype=torch.bfloat16)
pipe.load_lora_weights(lora)
pipe.enable_model_cpu_offload()
os.makedirs('/content/gen', exist_ok=True)
for i, w in enumerate(words):
    ref = Image.open(f'/content/edit_pairs/test_reference/{i:02d}.png').convert('RGB')
    img = pipe(prompt=instruction, image=ref, height=512, width=512, num_inference_steps=4,
               generator=torch.Generator('cpu').manual_seed(i)).images[0]
    img.save(f'/content/gen/{i:02d}.png')
print('done')

In [ ]:
# 7) Read each result back with OCR (Tesseract is a weak judge of handwriting, so also look at the sheet below)
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin > /dev/null
!pip -q install pytesseract python-Levenshtein
import pytesseract, Levenshtein
ok = 0
for i, w in enumerate(words):
    got = pytesseract.image_to_string(Image.open(f'/content/gen/{i:02d}.png'), lang='hin', config='--psm 8').strip()
    cer = Levenshtein.distance(w, got) / max(len(w), 1)
    ok += cer <= 0.2
    print(f'{w:12s} -> {got:12s} error rate {cer:.2f}')
print(f'{ok}/{len(words)} words readable within 20% error')

In [ ]:
# 8) Contact sheet: printed input on the left, generated handwriting on the right (saved to Drive)
n = len(words)
sheet = Image.new('RGB', (2 * 256 * 2, 256 * ((n + 1) // 2)), 'white')
for i in range(n):
    r = Image.open(f'/content/edit_pairs/test_reference/{i:02d}.png').resize((256, 256))
    g = Image.open(f'/content/gen/{i:02d}.png').resize((256, 256))
    x, y = (i % 2) * 512, (i // 2) * 256
    sheet.paste(r, (x, y)); sheet.paste(g, (x + 256, y))
sheet.save('/content/drive/MyDrive/hwdv1_edit_sheet.png')
sheet